# Data splitting (70% / 15% / 15%)

Run `data_cleaning.ipynb` first so `combined_dataset.csv` exists. This step does **not** change the essays — it only picks which row numbers belong to train, validation, or test.

| Split | Share | Use |
|-------|-------|-----|
| **Train** | 70% | Fit all models (XGBoost grid search uses cross-validation inside this fold) |
| **Validation** | 15% | Tune hyperparameters and pick the best setup |
| **Test** | 15% | Final evaluation only — same held-out rows for hybrid and every baseline |

We stratify on **label** (human vs AI) and **subject** (discipline) so each split keeps a similar class balance and discipline mix. If a discipline appears only once in a class, we align it with the smallest same-class discipline **for the split key only** (the saved CSV still shows the real subject). The same three index files are reused when slicing **H**, **S**, **E**, and **R** in `feature_extraction.ipynb`.

Outputs land in `data/processed/splits/`.

In [ ]:
# Paths and a small dry-run switch (match feature_extraction when testing).
import sys
from pathlib import Path

import pandas as pd

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR if (CURRENT_DIR / "data").exists() else CURRENT_DIR.parent.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

DATA_DIR = PROJECT_ROOT / "data"
COMBINED_CSV = DATA_DIR / "processed" / "combined_dataset.csv"
SPLITS_DIR = DATA_DIR / "processed" / "splits"

RUN_FULL_PIPELINE = True
RANDOM_STATE = 42
TRAIN_SIZE, VAL_SIZE, TEST_SIZE = 0.7, 0.15, 0.15

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Combined CSV exists:", COMBINED_CSV.exists())

## Load combined table

Row order here must match `row_index.csv` and every feature file from feature extraction (one row per essay, same order as the combined CSV).

In [ ]:
if not COMBINED_CSV.exists():
    raise FileNotFoundError(f"Run data_cleaning.ipynb first: {COMBINED_CSV}")

df = pd.read_csv(COMBINED_CSV)
expected_cols = {"text", "label", "source", "subject"}
missing = expected_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing columns: {missing}")

if not RUN_FULL_PIPELINE:
    df = df.head(200).copy()
    print("RUN_FULL_PIPELINE=False — using first 200 rows for a dry run.")

n_rows = len(df)
print("Rows:", n_rows)
print("Labels:\n", df["label"].value_counts())
print("Subjects (disciplines):", df["subject"].nunique())

## Stratified 70 / 15 / 15 split

First we hold out 15% for test, then split the rest into ~70% train and ~15% validation. Stratification keys are label + subject (discipline).

In [ ]:
from utils.features.data_splitting import (
    label_proportions_by_split,
    save_split_artifacts,
    split_balance_table,
    stratified_train_val_test_split,
)

idx_train, idx_val, idx_test = stratified_train_val_test_split(
    n_rows,
    df["label"].tolist(),
    df["subject"].tolist(),
    train_size=TRAIN_SIZE,
    val_size=VAL_SIZE,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)

print(
    "Split sizes:",
    len(idx_train),
    "train,",
    len(idx_val),
    "val,",
    len(idx_test),
    "test",
)
assert len(idx_train) + len(idx_val) + len(idx_test) == n_rows
assert len(set(idx_train) & set(idx_val)) == 0
assert len(set(idx_train) & set(idx_test)) == 0
assert len(set(idx_val) & set(idx_test)) == 0

## Check balance

Human vs AI proportions should stay close across train, val, and test. Subject counts will not match exactly in every discipline (some are rare), but stratification keeps large shifts out of any one fold.

In [ ]:
label_props = label_proportions_by_split(df, idx_train, idx_val, idx_test)
display(label_props)

# Full label × subject table — useful for the thesis appendix.
balance = split_balance_table(df, idx_train, idx_val, idx_test)
print("Strata rows:", len(balance))
balance.head(20)

## Save split indices

Downstream training loads `idx_train.npy`, `idx_val.npy`, and `idx_test.npy` from here. Do not refit models on validation or test rows.

In [ ]:
save_split_artifacts(
    SPLITS_DIR,
    idx_train,
    idx_val,
    idx_test,
    df=df,
    random_state=RANDOM_STATE,
    train_size=TRAIN_SIZE,
    val_size=VAL_SIZE,
    test_size=TEST_SIZE,
)

print("Saved splits to", SPLITS_DIR)
for name in ("idx_train.npy", "idx_val.npy", "idx_test.npy", "split_assignments.csv", "split_summary.json"):
    path = SPLITS_DIR / name
    print(f"  {name}: {path.exists()}")